# 01 - Exploratory Analysis

Signup growth, acquisition channels, funnel drop-off, and feature usage --
a visual companion to `database/queries/01_user_analysis.sql`,
`02_funnel_analysis.sql`, and `05_feature_analysis.sql`. All figures come
straight from Postgres; no logic is duplicated here that SQL already owns.

In [1]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

load_dotenv(find_dotenv())

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

# Fixed categorical order (never cycled) + single-hue sequential ramp,
# reused across every chart and the Streamlit dashboard for consistency.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQUENTIAL_BLUE = ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95"]

pd.options.display.float_format = "{:,.2f}".format

## Signup growth & acquisition

In [2]:
signups = pd.read_sql("""
    SELECT DATE_TRUNC('month', signup_date)::DATE AS signup_month, COUNT(*) AS new_users
    FROM users
    GROUP BY signup_month
    ORDER BY signup_month
""", engine)

fig = px.line(
    signups, x="signup_month", y="new_users", markers=True,
    color_discrete_sequence=[CATEGORICAL[0]],
    title="Monthly Signups",
)
fig.update_layout(template="plotly_white", yaxis_title="New users", xaxis_title=None)
fig.show()

In [3]:
channels = pd.read_sql("""
    SELECT acquisition_channel, COUNT(*) AS users
    FROM users
    GROUP BY acquisition_channel
    ORDER BY users DESC
""", engine)

fig = px.bar(
    channels, x="acquisition_channel", y="users",
    color="acquisition_channel", color_discrete_sequence=CATEGORICAL,
    title="Signups by Acquisition Channel",
)
fig.update_layout(template="plotly_white", showlegend=False, xaxis_title=None)
fig.show()

## Funnel drop-off

"Activated" is a derived state (>= 3 `report_export` uses within 7 days of
signup), computed here the same way as `database/queries/02_funnel_analysis.sql`
-- not stored as a raw event. See `docs/event_taxonomy.md`.

In [4]:
funnel = pd.read_sql("""
    WITH funnel_events AS (
        SELECT
            user_id,
            MAX(event_timestamp) FILTER (WHERE event_type = 'signup')               AS signup_at,
            MAX(event_timestamp) FILTER (WHERE event_type = 'email_verified')       AS verified_at,
            MAX(event_timestamp) FILTER (WHERE event_type = 'onboarding_completed') AS onboarded_at,
            MAX(event_timestamp) FILTER (WHERE event_type = 'trial_started')        AS trial_at,
            MAX(event_timestamp) FILTER (WHERE event_type = 'subscription_started') AS paid_at
        FROM events
        GROUP BY user_id
    ),
    key_feature_uses AS (
        SELECT user_id, event_timestamp
        FROM events
        WHERE event_type = 'feature_used' AND feature = 'report_export'
    ),
    activation AS (
        SELECT
            f.user_id,
            COUNT(k.event_timestamp) FILTER (
                WHERE k.event_timestamp BETWEEN f.signup_at AND f.signup_at + INTERVAL '7 days'
            ) >= 3 AS is_activated
        FROM funnel_events f
        LEFT JOIN key_feature_uses k ON k.user_id = f.user_id
        GROUP BY f.user_id, f.signup_at
    ),
    funnel AS (
        SELECT
            f.user_id,
            f.verified_at IS NOT NULL  AS reached_verified,
            f.onboarded_at IS NOT NULL AS reached_onboarded,
            a.is_activated             AS reached_activated,
            f.trial_at IS NOT NULL     AS reached_trial,
            f.paid_at IS NOT NULL      AS reached_paid
        FROM funnel_events f
        JOIN activation a ON a.user_id = f.user_id
    )
    SELECT 1 AS stage_order, 'signup' AS stage, COUNT(*) AS users FROM funnel
    UNION ALL SELECT 2, 'email_verified', COUNT(*) FILTER (WHERE reached_verified) FROM funnel
    UNION ALL SELECT 3, 'onboarding_completed', COUNT(*) FILTER (WHERE reached_onboarded) FROM funnel
    UNION ALL SELECT 4, 'activated', COUNT(*) FILTER (WHERE reached_activated) FROM funnel
    UNION ALL SELECT 5, 'trial_started', COUNT(*) FILTER (WHERE reached_trial) FROM funnel
    UNION ALL SELECT 6, 'subscription_started', COUNT(*) FILTER (WHERE reached_paid) FROM funnel
    ORDER BY stage_order
""", engine)

fig = go.Figure(go.Funnel(
    y=funnel["stage"], x=funnel["users"],
    marker={"color": CATEGORICAL[0]},
    textinfo="value+percent initial",
))
fig.update_layout(template="plotly_white", title="Signup-to-Paid Funnel")
fig.show()

## Feature usage & the power-user signal

`report_export` is the key feature seeded with a conversion/retention lift for
"power users" (>= 5 total uses) -- see `database/queries/05_feature_analysis.sql`
for the full discovery. Here we just look at adoption breadth.

In [5]:
feature_usage = pd.read_sql("""
    SELECT feature, COUNT(*) AS total_uses, COUNT(DISTINCT user_id) AS distinct_users
    FROM events
    WHERE event_type = 'feature_used'
    GROUP BY feature
    ORDER BY distinct_users DESC
""", engine)

fig = px.bar(
    feature_usage, x="feature", y="distinct_users",
    color="feature", color_discrete_sequence=CATEGORICAL,
    title="Feature Adoption (Distinct Users)",
)
fig.update_layout(template="plotly_white", showlegend=False, xaxis_title=None)
fig.show()

## Takeaways

- Signups grow steadily across the simulated window (Jan 2024 - Dec 2025), with
  no seasonal effects since the simulator doesn't model any.
- `organic` and `paid_ads` bring in the most users; see
  `01_user_analysis.sql` (1e) for how acquisition channel relates to
  *activation* rate, not just volume.
- The funnel narrows steeply after `onboarding_completed` -- only ~9% of
  signups ever convert to paid. `02_funnel_analysis.sql` breaks this down
  further by channel and by time-to-convert.
- `report_export` has far more distinct users at high usage intensity than
  any other feature -- the seeded power-user signal. `05_feature_analysis.sql`
  quantifies the resulting conversion and retention lift.